# Single cell disease relevance scores on meta cells 

I have generated new meta cells from samples, I will try to run scDRS for just insulin level traits.
I will test 100, 200, 500 and 1000 genes.
I have also added more covariates; donor, library_prep, cell_nuclei, dataset source, and number of meta cells

In [1]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pyhere import here      # Reproducible project paths
import gc

# dataframes
import pandas as pd
import numpy as np

# plotting
import matplotlib.pyplot as plt

# scanpy
import scanpy as sc

# GWAS
import gwaslab as gl

# Bash code 
import subprocess

sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import misc as mi

In [2]:
mi.create_directories(dir_path = str(here('data/scdrs/files')))
mi.create_directories(dir_path = str(here('data/scdrs/plots')))
mi.create_directories(dir_path = str(here('data/scdrs/objects')))
mi.create_directories(dir_path = str(here('data/scdrs/reference')))
mi.create_directories(dir_path = str(here('data/scdrs/sumstats_processed')))
mi.create_directories(dir_path = str(here('data/scdrs/sumstats_annotated')))

/work/islet_cartography_scrna/data/scdrs/files Directory already exists!
/work/islet_cartography_scrna/data/scdrs/plots Directory already exists!
/work/islet_cartography_scrna/data/scdrs/objects Directory already exists!
/work/islet_cartography_scrna/data/scdrs/reference Directory already exists!
/work/islet_cartography_scrna/data/scdrs/sumstats_processed Directory already exists!
/work/islet_cartography_scrna/data/scdrs/sumstats_annotated Directory already exists!


In [3]:
base_dir = str(here("data/scdrs"))
reference =  os.path.join(base_dir, "reference")
files_dir =  os.path.join(base_dir, "files")
sumsp_dir = os.path.join(base_dir, "sumstats_processed")
sumsa_dir = os.path.join(base_dir, "sumstats_annotated")

#### Prepare genesets for scdrs

In [ ]:
# ----------------------------- USER CONFIG -----------------------------------
# Work and reference directory
wd           = "/work/islet_cartography_scrna"
# Program paths
scdrs_bin    = f"{wd}/scrna_cartography_gwas/bin/scdrs"

gwas_dir     = f"{wd}/data/scdrs"
# File dir
pz_dir       = f"{gwas_dir}/pval_zscore"

# ----------------------------- KEEP INSULIN TRAIT ----------------------------
zscore = pd.read_csv(os.path.join(pz_dir, "zstat.tsv"), sep = "\t")
zscore = zscore[['SYMBOL', 'handedness', 'ins_protein_level', 'blood_insulin_level']]

zscore.to_csv(os.path.join(pz_dir, "zstat_insulin.tsv"), sep = "\t", index = False)

In [ ]:
# ----------------------------- USER CONFIG -----------------------------------
# Work and reference directory
wd           = "/work/islet_cartography_scrna"
# Program paths
scdrs_bin    = f"{wd}/scrna_cartography_gwas/bin/scdrs"

# File dir
pz_dir       = f"{gwas_dir}/pval_zscore"
gs_dir       = f"{gwas_dir}/gs_files"

os.makedirs(gs_dir, exist_ok=True)

n_genes = [100, 200, 500, 1000]
# ----------------------------- MUNGE -----------------------------------------
print(f"Selecting top genes using z-scores")

for n in n_genes:
    result = subprocess.run([
        scdrs_bin, "munge_gs", 
        "--out-file", f"{gs_dir}/insulin_sample_{n}.gs",
        "--zscore-file", f"{pz_dir}/zstat_insulin.tsv",
        "--fdr", "0.1",
        "--n-max", str(n),
    ], capture_output=True, text=True, check=True)
    
    print(f"Finished selecting top {n} genes using z-scores")

# Concatenate all .gs files with prefix, keeping only one header
with open(f"{gs_dir}/insulin_sample_combined.gs", "w") as outfile:
    for i, n in enumerate(n_genes):
        with open(f"{gs_dir}/insulin_sample_{n}.gs", "r") as infile:
            for line_num, line in enumerate(infile):
                # Skip header for all files except the first
                if line_num == 0 and i > 0:
                    continue
                
                parts = line.split("\t", 1)
                if len(parts) == 2 and line_num > 0:  # Don't modify header
                    trait, geneset = parts
                    outfile.write(f"{n}_{trait}\t{geneset}")
                else:
                    outfile.write(line)

In [ ]:
# Append with pop genes
ins = pd.read_csv(os.path.join(gs_dir, "insulin_sample_combined.gs"), sep = "\t")
pops = pd.read_csv(os.path.join(gs_dir, "pops_genes.gs"), sep = "\t")
pd.concat([ins, pops]).to_csv(os.path.join(gs_dir, "ins_and_pops.gs"), sep = "\t", index = False)

#### Disease enrichment scores per meta cell

First column: cell names, should be the same as adata.obs_names.

raw_score: raw disease score.

norm_score: normalized disease score.

mc_pval: cell-level MC p-value. Raw p-value without multiple testing adjustment.

pval: cell-level scDRS p-value. Raw p-value without multiple testing adjustment.

nlog10_pval: -log10(pval).

zscore: z-score converted from pval.

ctrl_raw_score_<i_ctrl> : raw control scores, specified by --flag_return_ctrl_raw_score True.

ctrl_norm_score_<i_ctrl> : normalized control scores, specified by --flag_return_ctrl_norm_score True.

##### Different number of insulin trait genes

In [ ]:
# ----------------------------- USER CONFIG -----------------------------------
# Work and reference directory
wd           = "/work/islet_cartography_scrna"
# Program paths
scdrs_bin    = f"{wd}/scrna_cartography_gwas/bin/scdrs"
h5ad_meta    = f"{wd}/data/meta_cell/objects/metacells_final_sample.h5ad"
gs_file      = f"{wd}/data/scdrs/gs_files/ins_and_pops.gs"
cov_file     = f"{wd}/data/meta_cell/files/metacell_cov_sample.tsv"
out_dir      = f"{wd}/data/scdrs/scdrs_metacell_results_sample"

os.makedirs(out_dir, exist_ok=True)
# ----------------------------- COMPUTE SCORES --------------------------------
print(f"Computing scores")

try:
    result = subprocess.run([
        scdrs_bin, "compute-score", 
        "--h5ad-file", h5ad_meta,
        "--h5ad-species", "human",
        "--gs-file", gs_file,
        "--gs-species", "human",
        "--cov-file", cov_file,
        "--flag-filter-data", "True",
        "--flag-raw-count", "True",
        "--n-ctrl", "1000",
        "--out-folder", out_dir 
    ], capture_output=True, text=True, check=True)
    
    print(f"Finished computing scores")
    print(result.stdout)
    
except subprocess.CalledProcessError as e:
    print(f"Command failed with exit code {e.returncode}")
    print(f"\nSTDOUT:\n{e.stdout}")
    print(f"\nSTDERR:\n{e.stderr}")
    # Use the actual command list instead of e.args
    print(f"\nCommand: {' '.join(str(arg) for arg in e.cmd)}")

#### Downstream analysis
#https://martinjzhang.github.io/scDRS/file_format.html#trait-scdrs-group-annot
For a given cell group-level annotation (e.g., tissue or cell type), assess cell group-disease association (control-score-based MC tests using 5% quantile) and within-cell group disease-association heterogeneity (control-score-based MC tests using Geary’s C).

#### Group level
For each annotation within the group and for each group of cell in the annotation, this function calculates:
The proportion of cells that has an significant association with the trait (FDR less than 0.05, 0.1 or 0.2)
The group level trait assocaition
The gorup level heterogenity (Here is uses the connectivity generated when finding neighbors)

MC = Monte Carlo
n_cell: number of cells from the cell group.

n_ctrl: number of control gene sets.

assoc_mcp: MC p-value for cell group-disease association. Raw p-value without multiple testing adjustment.

assoc_mcz: MC z-score for cell group-disease association.

hetero_mcp: MC p-value for within-cell group heterogeneity in association with disease. Raw p-value without multiple testing adjustment.

hetero_mcz: MC z-score for within-cell group heterogeneity in association with disease.

In [ ]:
import scanpy as sc

meta_ad = sc.read_h5ad("/work/islet_cartography_scrna/data/meta_cell/objects/metacells_final_sample.h5ad")

sc.pp.normalize_total(meta_ad, target_sum=1e4)
sc.pp.log1p(meta_ad)
sc.pp.highly_variable_genes(meta_ad, n_top_genes=2000)

sc.tl.pca(meta_ad, n_comps=30)
sc.pp.neighbors(meta_ad, n_neighbors=15, use_rep="X_pca")
sc.tl.umap(meta_ad)

meta_ad.write("/work/islet_cartography_scrna/data/meta_cell/objects/metacells_final_sample.h5ad") 

In [5]:
meta_ad = sc.read_h5ad("/work/islet_cartography_scrna/data/meta_cell/objects/metacells_final_sample.h5ad")
meta_ad

AnnData object with n_obs × n_vars = 9998 × 60656
    obs: 'manual_annotation', 'ethnicity_broad_harmonized', 'disease_harmonized', 'library_prep', 'ic_id_dataset', 'cell_nuclei', 'ic_id_platform_adjusted_sample', 'ic_id_donor_overall', 'cell_type', 'cell_type_disease', 'cell_type_sample'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts', 'raw'
    obsp: 'connectivities', 'distances'

In [ ]:
# Work and reference directory
wd           = "/work/islet_cartography_scrna"
# Program paths
scdrs_bin    = f"{wd}/scrna_cartography_gwas/bin/scdrs"
h5ad_meta    = f"{wd}/data/meta_cell/objects/metacells_final_sample.h5ad"
results_dir   = f"{wd}/data/scdrs/scdrs_metacell_results_sample"
out_dir      = f"{wd}/data/scdrs/scdrs_metacell_results_sample"

traits = pd.read_csv("/work/islet_cartography_scrna/data/scdrs/gs_files/ins_and_pops.gs", sep = "	")['TRAIT'].tolist()
groups = ['cell_type', 'cell_type_disease', 'cell_type_sample']
for group in groups:
    for trait in traits:
        print(f"Attempting {trait} for {group}")
        cmd = [
            scdrs_bin, "perform-downstream",
            "--h5ad-file", h5ad_meta,
            "--score-file", f"{results_dir}/{trait}.full_score.gz",
            "--out-folder", results_dir + "/",
            "--group-analysis", f"{group}",
            "--flag-filter-data", "True",
            "--flag-raw-count", "True"
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0:
            print(f" scDRS failed for {trait} ({group})")
            print("STDOUT:\n", result.stdout)
            print("STDERR:\n", result.stderr)
            continue
        else:
            print(f"Success for {trait} ({group})")

Attempting 100_blood_insulin_level for cell_type


#### Celltype

#### Combine results

In [ ]:
wd           = "/work/islet_cartography_scrna"

# Program paths
results_dir  = f"{wd}/data/scdrs/scdrs_metacell_results"

traits       = pd.read_csv("/work/islet_cartography_scrna/data/scdrs/gs_files/ins_and_pops.gs", sep = "	")['TRAIT'].tolist()

fdr_col      = 'n_fdr_0.1' 

output_dir   = f"{wd}/data/scdrs/scdrs_metacell_results_merged_sample"
os.makedirs(output_dir, exist_ok=True)


groups = ['cell_type', 'cell_type_disease', 'cell_type_sample']

for group in groups:
    data_list = []
    for trait in traits:
        file_path = f"{results_dir}/{trait}.scdrs_group.{group}"
        if os.path.exists(file_path):
            df = pd.read_csv(file_path, sep="\t", index_col=0)
            df['trait'] = trait
            data_list.append(df)
                  
    df_data = pd.concat(data_list, axis=0)
    df_data.to_csv(os.path.join(output_dir, f"group_analysis_merged_{group}"), index_label = "group")
